<a href="https://colab.research.google.com/github/temesgenaddise/cosc-650-applied-llm-systems/blob/main/Week-4/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4: Multi-Tool Assistant Assignment


This notebook uses Gemini Flash through Google's OpenAI-compatible endpoint. It defines three constrained tools, implements a complete request-and-response loop, provides a guarded arithmetic evaluator, records every tool call, evaluates the tools, and demonstrates structured recovery from a real invalid-date failure.


## Import modules and create the Gemini client



In [1]:
%pip install -q openai pandas requests

import ast
import json
import math
import operator
from datetime import date, timedelta

import pandas as pd
import requests
from openai import OpenAI
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("Add GEMINI_API_KEY to Colab Secrets before running the notebook.")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
MODEL = "gemini-3.8-flash"
print("Gemini client ready. Model:", MODEL)


Gemini client ready. Model: gemini-3.8-flash


# Part 1: Build the assistant

Part 1 defines three tools with explicit JSON types, required fields, enums, length constraints where useful, and `additionalProperties: false`.


In [2]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_weather",
            "description": "Return historical daily weather for one location on one exact past calendar date.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "minLength": 2,
                        "maxLength": 100,
                        "description": "City with optional state or country, such as Portland, Oregon, USA."
                    },
                    "day": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 31
                    },
                    "month": {
                        "type": "integer",
                        "minimum": 1,
                        "maximum": 12
                    },
                    "year": {
                        "type": "integer",
                        "minimum": 1940,
                        "maximum": 2100
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"]
                    }
                },
                "required": ["location", "day", "month", "year", "unit"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_temperature",
            "description": "Convert a numeric temperature between Celsius and Fahrenheit.",
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number"},
                    "from_unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"]
                    },
                    "to_unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"]
                    }
                },
                "required": ["value", "from_unit", "to_unit"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_safe_math",
            "description": "Evaluate a numeric arithmetic expression using only approved operators and math functions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "minLength": 1, "maxLength": 200},
                    "precision": {
                        "type": "integer",
                        "enum": [0, 2, 4, 6]
                    }
                },
                "required": ["expression", "precision"],
                "additionalProperties": False
            }
        }
    }
]


### Request-and-response loop

The loop sends the conversation and tool schemas to Gemini. When Gemini requests a tool, the code parses, validates, and executes the call, appends the structured result, and asks Gemini to continue. It stops when Gemini returns a final answer or reaches the step limit. The current date is supplied explicitly so relative dates such as “yesterday” can be resolved consistently during that run.


In [3]:
CURRENT_DATE = date.today()
SYSTEM_PROMPT = f"""
You are a careful tool-using assistant. Today's date is {CURRENT_DATE.isoformat()}.
Use tools whenever the user asks for weather, temperature conversion, or arithmetic.
Never invent tool results.
If a tool returns ok=false and recoverable=true, read the structured error, correct the call, and retry.
If recoverable=false, explain the failure without repeatedly calling the tool.
When multiple tools are needed, use the result of the first tool as input to the next.
Keep the final response concise. For historical weather, require an exact day, month, and year.
If the user gives a relative date, resolve it from today's date before calling the tool.
Report the resolved location, exact date, daily values, and source returned by the weather tool.
""".strip()

LIVE_MODEL_CALLS = 0

def run_agent(user_query: str, max_steps: int = 8) -> dict:
    global LIVE_MODEL_CALLS
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]
    call_log = []

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0
        )
        LIVE_MODEL_CALLS += 1
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        if not assistant_message.tool_calls:
            return {
                "query": user_query,
                "final_answer": assistant_message.content,
                "tool_call_log": call_log,
                "steps": step
            }

        for tool_call in assistant_message.tool_calls:
            try:
                arguments = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as error:
                arguments = tool_call.function.arguments
                tool_result = {
                    "ok": False,
                    "error": {
                        "error_type": "JSONDecodeError",
                        "message": str(error),
                        "recoverable": True,
                        "instruction": "Return one valid flat JSON object and retry."
                    }
                }
                call_log.append({
                    "call_id": tool_call.id,
                    "tool": tool_call.function.name,
                    "arguments": arguments,
                    "succeeded": False,
                    "error": tool_result["error"]
                })
            else:
                tool_result = execute_tool(
                    tool_call.function.name, arguments, tool_call.id, call_log
                )

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result)
            })

    return {
        "query": user_query,
        "final_answer": "Stopped safely because the maximum number of steps was reached.",
        "tool_call_log": call_log,
        "steps": max_steps
    }

print("Gemini function-calling loop defined for current date:", CURRENT_DATE.isoformat())


Gemini function-calling loop defined for current date: 2026-09-19


### Implement the weather and conversion tools

lookup_weather` validates an exact past calendar date, geocodes the location, and retrieves daily conditions. Very recent past dates use Open-Meteo's Forecast API; older dates use its Historical Weather Archive API. The result includes daily mean, minimum, and maximum temperatures, precipitation, maximum wind speed, a weather description, the resolved location, the exact date, and the source.

The tool depends on internet access. Because the geocoder returns the best single match, users should include a state or country when a city name is ambiguous.


In [4]:
WMO_WEATHER_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "depositing rime fog",
    51: "light drizzle", 53: "moderate drizzle", 55: "dense drizzle",
    56: "light freezing drizzle", 57: "dense freezing drizzle",
    61: "slight rain", 63: "moderate rain", 65: "heavy rain",
    66: "light freezing rain", 67: "heavy freezing rain",
    71: "slight snow", 73: "moderate snow", 75: "heavy snow", 77: "snow grains",
    80: "slight rain showers", 81: "moderate rain showers", 82: "violent rain showers",
    85: "slight snow showers", 86: "heavy snow showers",
    95: "thunderstorm", 96: "thunderstorm with slight hail", 99: "thunderstorm with heavy hail"
}

def lookup_weather(location: str, day: int, month: int, year: int, unit: str) -> dict:
    if unit not in {"celsius", "fahrenheit"}:
        raise ValueError("unit must be 'celsius' or 'fahrenheit'.")

    location = location.strip()
    if not 2 <= len(location) <= 100:
        raise ValueError("location must contain 2 to 100 characters.")

    try:
        requested_date = date(year, month, day)
    except ValueError as error:
        raise ValueError(f"Invalid calendar date: {year:04d}-{month:02d}-{day:02d}") from error

    today = date.today()
    if requested_date >= today:
        raise ValueError("The requested date must be before today.")
    if requested_date < date(1940, 1, 1):
        raise ValueError("Historical weather is available from January 1, 1940.")

    iso_date = requested_date.isoformat()

    try:
        geo_response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": location, "count": 1, "language": "en", "format": "json"},
            timeout=10
        )
        geo_response.raise_for_status()
        matches = geo_response.json().get("results", [])
        if not matches:
            raise ValueError(f"No location found for: {location}")
        place = matches[0]

        # The Forecast API reliably supports yesterday and other very recent days.
        # The Archive API supplies older reanalysis data dating back to 1940.
        recent_cutoff = today - timedelta(days=5)
        if requested_date >= recent_cutoff:
            weather_url = "https://api.open-meteo.com/v1/forecast"
            source_name = "Open-Meteo Forecast API historical day"
        else:
            weather_url = "https://archive-api.open-meteo.com/v1/archive"
            source_name = "Open-Meteo Historical Weather API"

        weather_response = requests.get(
            weather_url,
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "start_date": iso_date,
                "end_date": iso_date,
                "daily": (
                    "weather_code,temperature_2m_mean,temperature_2m_max,"
                    "temperature_2m_min,precipitation_sum,wind_speed_10m_max"
                ),
                "temperature_unit": unit,
                "wind_speed_unit": "mph" if unit == "fahrenheit" else "kmh",
                "precipitation_unit": "inch" if unit == "fahrenheit" else "mm",
                "timezone": "auto"
            },
            timeout=10
        )
        weather_response.raise_for_status()
        daily = weather_response.json().get("daily", {})
        if not daily.get("time"):
            raise RuntimeError(f"No historical weather returned for {iso_date}.")
    except requests.RequestException as error:
        raise RuntimeError(f"Historical weather request failed: {error}") from error

    resolved_parts = [place.get("name"), place.get("admin1"), place.get("country")]
    resolved_location = ", ".join(part for part in resolved_parts if part)
    temperature_symbol = "°F" if unit == "fahrenheit" else "°C"
    wind_unit = "mph" if unit == "fahrenheit" else "km/h"
    precipitation_unit = "inches" if unit == "fahrenheit" else "mm"
    weather_code = daily["weather_code"][0]

    return {
        "requested_location": location,
        "resolved_location": resolved_location,
        "date": daily["time"][0],
        "latitude": place["latitude"],
        "longitude": place["longitude"],
        "condition": WMO_WEATHER_CODES.get(weather_code, "unknown condition"),
        "weather_code": weather_code,
        "mean_temperature": daily["temperature_2m_mean"][0],
        "minimum_temperature": daily["temperature_2m_min"][0],
        "maximum_temperature": daily["temperature_2m_max"][0],
        "temperature_unit": temperature_symbol,
        "precipitation_sum": daily["precipitation_sum"][0],
        "precipitation_unit": precipitation_unit,
        "maximum_wind_speed": daily["wind_speed_10m_max"][0],
        "wind_speed_unit": wind_unit,
        "source": source_name
    }

def convert_temperature(value: float, from_unit: str, to_unit: str) -> dict:
    valid = {"celsius", "fahrenheit"}
    if from_unit not in valid or to_unit not in valid:
        raise ValueError("Units must be 'celsius' or 'fahrenheit'.")
    if from_unit == to_unit:
        converted = value
    elif from_unit == "celsius":
        converted = value * 9 / 5 + 32
    else:
        converted = (value - 32) * 5 / 9
    return {
        "input_value": value,
        "from_unit": from_unit,
        "to_unit": to_unit,
        "converted_value": round(converted, 2)
    }

# Example exact historical date.
print(lookup_weather("Portland, Oregon, USA", day=28, month=2, year=2026, unit="celsius"))
print(convert_temperature(18, "celsius", "fahrenheit"))


{'requested_location': 'Portland, Oregon, USA', 'resolved_location': 'Portland, Oregon, United States', 'date': '2026-02-28', 'latitude': 45.52345, 'longitude': -122.67621, 'condition': 'overcast', 'weather_code': 3, 'mean_temperature': 6.3, 'minimum_temperature': 2.1, 'maximum_temperature': 11.5, 'temperature_unit': '°C', 'precipitation_sum': 0.0, 'precipitation_unit': 'mm', 'maximum_wind_speed': 9.4, 'wind_speed_unit': 'km/h', 'source': 'Open-Meteo Historical Weather API'}
{'input_value': 18, 'from_unit': 'celsius', 'to_unit': 'fahrenheit', 'converted_value': 64.4}


**Interpretation:** The first output reports historical weather for Portland on February 28, 2026. The second output correctly converts 18 °C to 64.4 °F.


# Part 2: Add a guarded arithmetic tool

**Permitted:** numeric constants; parentheses; `+`, `-`, `*`, `/`, `//`, `%`, and `**`; unary signs; and the allowlisted functions `sqrt`, `sin`, `cos`, `tan`, `log`, `log10`, `exp`, `floor`, `ceil`, `abs`, and `round`.

**Blocked:** imports, filesystem and network access, process execution, attribute access, variables, assignment, loops, comprehensions, user-defined functions/classes, keyword arguments, and non-allowlisted functions. The evaluator also restricts expression length, AST size, exponent size, and result magnitude.


In [5]:
ALLOWED_BINARY = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod, ast.Pow: operator.pow
}
ALLOWED_UNARY = {ast.UAdd: operator.pos, ast.USub: operator.neg}
ALLOWED_FUNCTIONS = {
    "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "tan": math.tan,
    "log": math.log, "log10": math.log10, "exp": math.exp,
    "floor": math.floor, "ceil": math.ceil, "abs": abs, "round": round
}
MAX_EXPRESSION_LENGTH = 200
MAX_AST_NODES = 100
MAX_ABS_VALUE = 1e100
MAX_POWER = 100

def _check_numeric_result(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError("The expression must produce a real numeric result.")
    if not math.isfinite(float(value)) or abs(value) > MAX_ABS_VALUE:
        raise ValueError("Result exceeds the allowed numeric range.")
    return value

def _evaluate_node(node):
    if isinstance(node, ast.Expression):
        return _evaluate_node(node.body)
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return _check_numeric_result(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_BINARY:
        if isinstance(node.op, ast.Pow):
            right = _evaluate_node(node.right)
            if abs(right) > MAX_POWER:
                raise ValueError("Exponent exceeds the allowed limit.")
            left = _evaluate_node(node.left)
        else:
            left = _evaluate_node(node.left)
            right = _evaluate_node(node.right)
        return _check_numeric_result(ALLOWED_BINARY[type(node.op)](left, right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_UNARY:
        return _check_numeric_result(ALLOWED_UNARY[type(node.op)](_evaluate_node(node.operand)))
    if isinstance(node, ast.Call):
        if not isinstance(node.func, ast.Name) or node.func.id not in ALLOWED_FUNCTIONS:
            raise ValueError("Function is not on the allowlist.")
        if node.keywords:
            raise ValueError("Keyword arguments are blocked.")
        arguments = [_evaluate_node(arg) for arg in node.args]
        return _check_numeric_result(ALLOWED_FUNCTIONS[node.func.id](*arguments))
    raise ValueError(f"Blocked syntax category: {type(node).__name__}")

def _checked_math(expression: str):
    if not expression.strip():
        raise ValueError("Expression cannot be empty.")
    if len(expression) > MAX_EXPRESSION_LENGTH:
        raise ValueError("Expression is too long.")
    tree = ast.parse(expression, mode="eval")
    if sum(1 for _ in ast.walk(tree)) > MAX_AST_NODES:
        raise ValueError("Expression contains too many syntax nodes.")
    return _evaluate_node(tree)

def run_safe_math(expression: str, precision: int) -> dict:
    if precision not in {0, 2, 4, 6}:
        raise ValueError("Precision must be one of 0, 2, 4, or 6.")
    value = _checked_math(expression)
    return {
        "expression": expression,
        "result": round(value, precision),
        "precision": precision
    }

print(run_safe_math("sqrt(144) + 3**2", 2))
try:
    run_safe_math("__import__('os').system('whoami')", 2)
except Exception as error:
    print("Blocked test:", error)


{'expression': 'sqrt(144) + 3**2', 'result': 21.0, 'precision': 2}
Blocked test: Function is not on the allowlist.


**Interpretation:** The permitted expression returns 21.0. The unsafe import/process expression is rejected before execution because attribute access and non-allowlisted functions are blocked.

### Add dispatch, complete validation, and structured logging

This cell maps tool names to Python functions. validate_arguments mirrors the schema's required fields, additional-property rule, types, enums, numeric bounds, and string-length bounds. execute_tool converts failures into structured errors and marks external request failures as non-recoverable by argument correction.


In [6]:
TOOL_FUNCTIONS = {
    "lookup_weather": lookup_weather,
    "convert_temperature": convert_temperature,
    "run_safe_math": run_safe_math
}

def validate_arguments(tool_name: str, args: dict):
    if not isinstance(args, dict):
        raise TypeError("Tool arguments must be one flat JSON object.")

    schema = next(
        tool["function"]["parameters"] for tool in TOOLS
        if tool["function"]["name"] == tool_name
    )
    properties = schema["properties"]
    missing = [name for name in schema["required"] if name not in args]
    extra = [name for name in args if name not in properties]
    if missing:
        raise ValueError(f"Missing required fields: {missing}")
    if extra:
        raise ValueError(f"Unexpected fields: {extra}")

    for name, value in args.items():
        rule = properties[name]
        expected = rule["type"]
        type_ok = {
            "string": isinstance(value, str),
            "number": isinstance(value, (int, float)) and not isinstance(value, bool),
            "integer": isinstance(value, int) and not isinstance(value, bool)
        }[expected]
        if not type_ok:
            raise TypeError(f"{name} must have type {expected}.")
        if "enum" in rule and value not in rule["enum"]:
            raise ValueError(f"{name} must be one of {rule['enum']}.")
        if "minimum" in rule and value < rule["minimum"]:
            raise ValueError(f"{name} must be at least {rule['minimum']}.")
        if "maximum" in rule and value > rule["maximum"]:
            raise ValueError(f"{name} must be at most {rule['maximum']}.")
        if "minLength" in rule and len(value) < rule["minLength"]:
            raise ValueError(f"{name} must contain at least {rule['minLength']} characters.")
        if "maxLength" in rule and len(value) > rule["maxLength"]:
            raise ValueError(f"{name} must contain at most {rule['maxLength']} characters.")

def execute_tool(tool_name: str, args, call_id: str, log: list) -> dict:
    entry = {"call_id": call_id, "tool": tool_name, "arguments": args}
    try:
        if tool_name not in TOOL_FUNCTIONS:
            raise ValueError(f"Unknown tool: {tool_name}")
        validate_arguments(tool_name, args)
        output = TOOL_FUNCTIONS[tool_name](**args)
        entry.update({"succeeded": True, "result": output})
        response = {"ok": True, "result": output}
    except Exception as error:
        recoverable = not isinstance(error, requests.RequestException) and not (
            isinstance(error, RuntimeError) and "request failed" in str(error).lower()
        )
        structured_error = {
            "error_type": type(error).__name__,
            "message": str(error),
            "recoverable": recoverable,
            "instruction": (
                "Correct the arguments or expression and retry."
                if recoverable else
                "Do not retry repeatedly; report the external-service failure."
            )
        }
        entry.update({"succeeded": False, "error": structured_error})
        response = {"ok": False, "error": structured_error}
    log.append(entry)
    return response

test_log = []
print(execute_tool(
    "convert_temperature",
    {"value": 32, "from_unit": "fahrenheit", "to_unit": "celsius"},
    "direct_conversion_test",
    test_log
))

print(execute_tool(
    "lookup_weather",
    {"location": "Portland, Oregon, USA", "day": 30, "month": 2, "year": 2026, "unit": "celsius"},
    "invalid_date_test",
    test_log
))

print(execute_tool(
    "lookup_weather",
    {"location": "Portland, Oregon, USA", "day": 28, "month": 2, "year": 2026, "unit": "celsius"},
    "corrected_date_retry",
    test_log
))


{'ok': True, 'result': {'input_value': 32, 'from_unit': 'fahrenheit', 'to_unit': 'celsius', 'converted_value': 0.0}}
{'ok': False, 'error': {'error_type': 'ValueError', 'message': 'Invalid calendar date: 2026-02-30', 'recoverable': True, 'instruction': 'Correct the arguments or expression and retry.'}}
{'ok': True, 'result': {'requested_location': 'Portland, Oregon, USA', 'resolved_location': 'Portland, Oregon, United States', 'date': '2026-02-28', 'latitude': 45.52345, 'longitude': -122.67621, 'condition': 'overcast', 'weather_code': 3, 'mean_temperature': 6.3, 'minimum_temperature': 2.1, 'maximum_temperature': 11.5, 'temperature_unit': '°C', 'precipitation_sum': 0.0, 'precipitation_unit': 'mm', 'maximum_wind_speed': 9.4, 'wind_speed_unit': 'km/h', 'source': 'Open-Meteo Historical Weather API'}}


**Interpretation:** The first call successfully converts 32 °F to 0 °C. The second call passes the flat schema checks but fails runtime calendar validation because February 30 does not exist. The third call demonstrates recovery with the corrected date February 28, 2026. Each call has a unique identifier in the log.


# Part 3: Evaluate


In [7]:
def show_run(run: dict):
    print("QUERY:", run["query"])
    print("\nFINAL ANSWER:", run["final_answer"])
    print("\nTOOL-CALL LOG")
    if not run["tool_call_log"]:
        print("  No tools called.")
    for index, entry in enumerate(run["tool_call_log"], start=1):
        print(f"  {index}. tool={entry['tool']}")
        print(f"     arguments={json.dumps(entry['arguments'], ensure_ascii=False)}")
        print(f"     succeeded={entry['succeeded']}")
        if entry["succeeded"]:
            print(f"     result={json.dumps(entry['result'], ensure_ascii=False)}")
        else:
            print(f"     error={json.dumps(entry['error'], ensure_ascii=False)}")

## Run three evaluation queries

The three queries exercise every tool:

1. Query 1 asks for “yesterday” without inserting the exact date into the user message. Gemini must resolve it using the current date supplied in the system prompt.
2. Query 2 calls the guarded arithmetic tool.
3. Query 3 calls the weather tool and then passes its returned mean temperature to the conversion tool.

Each query runs once to avoid unnecessary external API calls. A summary table reports tool calls, successful calls, failed calls, and whether the agent returned a final answer.


In [ ]:
last_month_day = (CURRENT_DATE.replace(day=1) - timedelta(days=1)).replace(day=15)

EVALUATION_QUERIES = [
    "What was the weather in Portland, Oregon, USA yesterday? Use Celsius.",
    "Use the safe math tool to calculate sqrt(225) + 7 * 3. Round to 2 decimals.",
    (
        "Look up the mean temperature in Portland, Oregon, USA on "
        f"{last_month_day.strftime('%B %d, %Y')} in Celsius, then convert that returned "
        "mean temperature to Fahrenheit."
    )
]

all_runs = []
for query_number, query in enumerate(EVALUATION_QUERIES, start=1):
    run = run_agent(query)
    run["query_number"] = query_number
    all_runs.append(run)
    print(f"\nQUERY {query_number}")
    show_run(run)

evaluation_summary = pd.DataFrame([
    {
        "query": run["query_number"],
        "agent_steps": run["steps"],
        "tool_calls": len(run["tool_call_log"]),
        "successful_calls": sum(item["succeeded"] for item in run["tool_call_log"]),
        "failed_calls": sum(not item["succeeded"] for item in run["tool_call_log"]),
        "final_answer_returned": bool(run["final_answer"])
    }
    for run in all_runs
])

print("\nEVALUATION SUMMARY")
display(evaluation_summary)


# Ask Review //////////////////


QUERY 1
QUERY: What was the weather in Portland, Oregon, USA yesterday? Use Celsius.

FINAL ANSWER: **Weather Summary for Portland, Oregon:**

* **Resolved Location:** Portland, Oregon, United States
* **Date:** September 18, 2026
* **Condition:** Fog
* **Mean Temperature:** 15.5°C (Min: 12.1°C, Max: 19.0°C)
* **Precipitation:** 0.0 mm
* **Max Wind Speed:** 8.4 km/h
* **Source:** Open-Meteo Forecast API historical day

TOOL-CALL LOG
  1. tool=lookup_weather
     arguments={"unit": "celsius", "month": 9, "day": 18, "location": "Portland, Oregon, USA", "year": 2026}
     succeeded=True
     result={"requested_location": "Portland, Oregon, USA", "resolved_location": "Portland, Oregon, United States", "date": "2026-09-18", "latitude": 45.52345, "longitude": -122.67621, "condition": "fog", "weather_code": 45, "mean_temperature": 15.5, "minimum_temperature": 12.1, "maximum_temperature": 19.0, "temperature_unit": "°C", "precipitation_sum": 0.0, "precipitation_unit": "mm", "maximum_wind_speed

,query,agent_steps,tool_calls,successful_calls,failed_calls,final_answer_returned
0,1,2,1,1,0,True
1,2,2,1,1,0,True
2,3,3,2,2,0,True


**Result interpretation:**

- **Query 1:** Gemini resolved “yesterday” from the current date in the system prompt and called lookup_weather` with an exact day, month, and year.
- **Query 2:** Gemini called run_safe_math`; the tool returned 36.0 at two-decimal precision.
- **Query 3:** Gemini called lookup_weather, extracted the returned mean temperature, and passed it to `convert_temperature.

# Part 4: Find One Failure and Explain it

The demonstrated failure is an **invalid calendar-date runtime validation error**. The lookup_weather call receives February 30, 2026. Its object structure, required fields, field types, numeric ranges, and unit all satisfy the JSON schema, but the combined date does not exist. Python's date constructor detects this semantic problem inside the tool.


### Schema involved

In [9]:
{
    "type": "object",
    "properties": {
        "location": {"type": "string", "minLength": 2, "maxLength": 100},
        "day": {"type": "integer", "minimum": 1, "maximum": 31},
        "month": {"type": "integer", "minimum": 1, "maximum": 12},
        "year": {"type": "integer", "minimum": 1940, "maximum": 2100},
        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
    },
    "required": ["location", "day", "month", "year", "unit"],
    "additionalProperties": False
}

{'type': 'object',
 'properties': {'location': {'type': 'string',
   'minLength': 2,
   'maxLength': 100},
  'day': {'type': 'integer', 'minimum': 1, 'maximum': 31},
  'month': {'type': 'integer', 'minimum': 1, 'maximum': 12},
  'year': {'type': 'integer', 'minimum': 1940, 'maximum': 2100},
  'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}},
 'required': ['location', 'day', 'month', 'year', 'unit'],
 'additionalProperties': False}

The schema restricts day to 1–31 and month to 1–12, but independent numeric ranges cannot express every relationship between day, month, leap year, and year. Consequently, day=30 and month=2 are independently valid schema values even though February 30 is not a real date. This is why runtime calendar validation remains necessary.


### Bad call, why it failed, and the fix

In [10]:
failure_recovery_log = []

# Bad call: structurally valid, but February 30 does not exist.
bad_call_arguments = {
    "location": "Portland, Oregon, USA",
    "day": 30,
    "month": 2,
    "year": 2026,
    "unit": "celsius"
}
bad_result = execute_tool(
    "lookup_weather", bad_call_arguments, "part4_invalid_date", failure_recovery_log
)
print("BAD CALL RESULT")
print(json.dumps(bad_result, indent=2))

# Recovery: change only the invalid date to the final valid day of February 2026.
corrected_call_arguments = {**bad_call_arguments, "day": 28}
corrected_result = execute_tool(
    "lookup_weather", corrected_call_arguments, "part4_corrected_date", failure_recovery_log
)
print("\nCORRECTED RETRY RESULT")
print(json.dumps(corrected_result, indent=2))


BAD CALL RESULT
{
  "ok": false,
  "error": {
    "error_type": "ValueError",
    "message": "Invalid calendar date: 2026-02-30",
    "recoverable": true,
    "instruction": "Correct the arguments or expression and retry."
  }
}

CORRECTED RETRY RESULT
{
  "ok": true,
  "result": {
    "requested_location": "Portland, Oregon, USA",
    "resolved_location": "Portland, Oregon, United States",
    "date": "2026-02-28",
    "latitude": 45.52345,
    "longitude": -122.67621,
    "condition": "overcast",
    "weather_code": 3,
    "mean_temperature": 6.3,
    "minimum_temperature": 2.1,
    "maximum_temperature": 11.5,
    "temperature_unit": "\u00b0C",
    "precipitation_sum": 0.0,
    "precipitation_unit": "mm",
    "maximum_wind_speed": 9.4,
    "wind_speed_unit": "km/h",
    "source": "Open-Meteo Historical Weather API"
  }
}


### Why it failed and how recovery worked

The bad object is flat, contains all required fields, uses the correct types, stays within the schema's numeric ranges, and supplies an allowed unit. Nevertheless, date(2026, 2, 30) raises ValueError because February 30 does not exist. lookup_weather converts this into the clearer message Invalid calendar date: 2026-02-30, and execute_tool returns a structured response containing ok: false`, `recoverable: true`, and an instruction to correct the arguments.

The recovery changes only day from 30 to 28. The corrected call returns ok: true` and historical weather data. This shows an important boundary: the JSON schema validates individual fields, while the Python tool validates relationships among those fields.




# Part 5: Submit
Open a pull request in your course repository. In the description, briefly explain your schema design choices and include links to your notebook and an issue documenting the failure and recovery from Part 4.